## stellar parameters

In [ ]:
import pandas as pd

df = pd.read_parquet('../data/Zhang2023/Zhang2023_lamost.parquet')
df.shape

In [ ]:
# ID = 5239758155778687360 #TOI-6715
ID = 1935439776865267072 #TOI-5973
df.gaia_dr3_id.isin([ID]).sum()

In [ ]:
type(df['gaia_dr3_id'].iloc[0])

In [ ]:
df.query("gaia_dr3_id==@ID")

## download and merge

https://zenodo.org/records/7811871

In [ ]:
import h5py

fp = "../data/Zhang2023/stellar_params_catalog_00.h5"

with h5py.File(fp, 'r') as f:
    print("Keys at the root level:")
    for key in f.keys():
        print(key)

In [ ]:
with h5py.File(fp, 'r') as f:
    try:
        params = f['stellar_params_est'][()]
        err = f['stellar_params_err'][()]
        qf = f['quality_flags'][()]
        ra = f['ra'][()]
        dec = f['dec'][()]
        gid = f['gdr3_source_id'][()]
    except Exception as e:
        print(e)

In [ ]:
import pandas as pd
#effective temperature in kiloKelvin, [Fe/H] in dex, log(g) in dex, E in mag, parallax in mas
col_names = ['teff_K','feh_dex','logg_dex','extinction_mag','parallax_mas']
df = pd.DataFrame(params, columns=col_names)
df['teff_K'] *= 1e3
df['ra'] = ra
df['dec'] = dec
df['gaia_dr3_id'] = gid
df['quality_flag'] = qf
df.head()

In [ ]:
from glob import glob

h5_files = sorted(list(glob("../data/Zhang2023/stellar_params_catalog_*.h5")))
h5_files

In [ ]:
from astropy.table import Table
from tqdm import tqdm

for fp in tqdm(h5_files[:-1]):
    print(f"Loading {fp}")
    output = Table()
    with h5py.File(fp, 'r') as f:
        for i, key in enumerate(f.keys()):
            #print(f"Loading {i+1}/{len(f.keys())}: {key}")
            output[key] = f[key][:]

In [ ]:
output['stellar_params_est'].value.shape

In [ ]:
import pandas as pd

#effective temperature in kiloKelvin, [Fe/H] in dex, log(g) in dex, E in mag, parallax in mas
col_names = ['teff_K','feh_dex','logg_dex','extinction_mag','parallax_mas']
df = pd.DataFrame(output['stellar_params_est'].value, columns=col_names)
df['teff_K'] *= 1e3
df['ra'] = output['ra'].value
df['dec'] = output['dec'].value
df['gaia_dr3_id'] = output['gdr3_source_id'].value
df['quality_flag'] = output['quality_flags'].value
df.head()

In [ ]:
df.shape

In [ ]:
# ID = 5239758155778687360 #TOI-6715
ID = 1935439776865267072 #TOI-5973
df.gaia_dr3_id.isin([ID]).sum()

In [ ]:
df.query("gaia_dr3_id==@ID")

In [ ]:
df2 = df[df['quality_flag']<8]
df2.shape

In [ ]:
df.to_parquet('../data/Zhang2023/Zhang2023_lamost.parquet')

In [ ]:
!ls -lth '../data/Zhang2023/Zhang2023_lamost.parquet'

## lamost spectra

In [ ]:
from astroquery.esasky import ESASky

spectra_list = ESASky.list_spectra()
spectra_list

In [ ]:
result = ESASky.query_object_catalogs(position="TOI-5973", catalogs="LAMOST_MRS")
result

In [ ]:
result[0]

In [ ]:
result[0]['product_url'][-1]

In [ ]:
import requests

url = result[0]['product_url'][0] #choose the first spectra

data = requests.get(url, timeout=30)
with open("lamost_spectrum.fits", "wb") as f:
    f.write(data.content)

In [ ]:
from astropy.io import fits
hdul = fits.open("lamost_spectrum.fits")
hdul.info()

wave = hdul[1].data['wavelength']
flux = hdul[1].data['flux']